# Crop a GRS L2A image and export the subset

This notebook extracts an area of interest (AOI) from a GRS **L2A** product and writes it as a new,
smaller L2A product that can be reopened with `grstbx.L2grs` like any GRS image.

Workflow:

1. **Settings**: input product, output directory and format, default AOI;
2. **Preview** the image at a coarse resolution (pyramid level) and **draw the AOI** on an interactive map;
3. **Crop** the full-resolution product to the AOI: all variables, flags and the ancillary data;
4. **Export** to NetCDF or Zarr with the GRS packing, plus an optional GeoTIFF of $R_{rs}$;
5. **Check** the exported product, then optionally apply the same AOI to a series of images.

The crop and export functions are part of grstbx (module `grstbx.subset`):
`grstbx.open_l2a`, `grstbx.aoi_from_box`, `grstbx.crop_l2a`, `grstbx.export_l2a`,
`grstbx.export_rrs_geotiff`, `grstbx.subset_path` and `grstbx.crop_and_export`.
The maps use `grstbx.plot_rgb` (true-colour composite with the AOI outline) and `grstbx.utm_projection`.

**Performance**

- The preview and the drawing use a coarse pyramid level and only the three true-colour bands,
  so a whole tile is displayed in a few seconds.
- The crop is lazy: `rio.clip_box` slices the dask arrays, so only the chunks intersecting the AOI
  are read from disk, whatever the size of the tile.
- The pixels outside a non-rectangular AOI are masked on the small subset only (`rasterio`
  rasterization). The integer `flags` and `mask` keep their dtype, unlike with `rio.clip`, which
  converts them to float.

**Input:** a GRS L2A product, either a Zarr store (`*.zarr`, single resolution or multiscale pyramid
with an `ancillary` group) or a folder `<name>/` holding `<name>.nc` and `<name>_anc.nc`.

## 1. Libraries

In [ ]:
import os
import glob

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import panel as pn

import grs
import grstbx
from grstbx import visual, plot_rgb, utm_projection

pn.extension()
opj = os.path.join

print(f'grs:    {grs.__version__}')
print(f'grstbx: {grstbx.__version__}')

## 2. Settings

All the parameters are gathered in the next cell:

| parameter | description |
|---|---|
| `FILE` | input L2A product (`.zarr` store or NetCDF folder) |
| `SITE` | name of the AOI, appended to the output file names |
| `ODIR` | output directory; each subset is written in `ODIR/SITE/` |
| `FMT` | `'netcdf'` (`<name>.nc` + `<name>_anc.nc` in a folder `<name>/`) or `'zarr'` (`<name>.zarr`, with pyramids) |
| `PREVIEW_LEVEL` | pyramid level used for the preview and the drawing (e.g. 2 = 4x coarser); ignored for single-resolution products |
| `MASK_OUTSIDE` | for a non-rectangular AOI, mask the pixels outside the polygon (otherwise its bounding box is kept as is) |
| `AOI_FILE` | optional vector file (GeoJSON, shapefile, GeoPackage...) defining the AOI |
| `CENTER_LONLAT`, `BOX_SIZE_M` | default AOI: rectangle of `BOX_SIZE_M` (width, height in m) centred on (lon, lat) |

The AOI actually used is, by order of priority: the **polygon drawn** on the map of section 3, the
polygons of `AOI_FILE`, the default box.

In [ ]:
FILE = ('/data/satellite/Sentinel-2/test/'
        'S2A_MSIL2AGRS_20170412T110621_N0500_R137_T30TWT_20231111T110836_V3.0.1dev/'
        'S2A_MSIL2AGRS_20170412T110621_N0500_R137_T30TWT_20231111T110836_V3.0.1dev.zarr')

SITE = 'loire_estuary'
ODIR = '/data/tmp/grstbx_subset'
FMT = 'zarr'          # 'netcdf' or 'zarr'
PREVIEW_LEVEL = 2
MASK_OUTSIDE = True

AOI_FILE = None         # e.g. '/path/to/aoi.geojson'
CENTER_LONLAT = (-2.5, 47.12)
BOX_SIZE_M = (35000, 35000)

## 3. Preview and AOI drawing

`grstbx.open_l2a` opens the product lazily (nothing is loaded yet) together with its ancillary data
(CAMS fields and gaseous transmittance on a coarse ~10 km grid), which is exported with the subset.

In [ ]:
levels = grstbx.zarr_levels(FILE) if grstbx.driver.is_zarr(FILE) else []
preview_level = min(PREVIEW_LEVEL, len(levels) - 1) if levels else 0

l2a_prod, ancillary = grstbx.open_l2a(FILE)
preview, _ = grstbx.open_l2a(FILE, level=preview_level)
crs = l2a_prod.rio.crs
proj = utm_projection(crs)

print(f'pyramid levels: {levels or "single resolution"}')
for name, ds in [('full resolution', l2a_prod), (f'preview (level {preview_level})', preview)]:
    print(f'{name:22s}: {ds.sizes["x"]} x {ds.sizes["y"]} pixels, {abs(ds.rio.resolution()[0]):.0f} m')
print(f'CRS: {crs}, date: {l2a_prod.time.dt.date.values}')

Default AOI (from `AOI_FILE` or the box around `CENTER_LONLAT`), drawn in red on the preview:

In [ ]:
if AOI_FILE:
    default_aoi = gpd.read_file(AOI_FILE)
else:
    default_aoi = grstbx.aoi_from_box(*CENTER_LONLAT, width=BOX_SIZE_M[0], height=BOX_SIZE_M[1])

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': proj})
plot_rgb(preview, ax, aoi=default_aoi)
ax.set_title(f'Preview (level {preview_level}) and default AOI');

To use another AOI, draw it on the map below with the **Polygon Draw** tool of the toolbar:
double-click to start the polygon, click to add vertices, double-click to close it. Only the **last**
polygon drawn is used. Then run the next cells. Without a drawing, the default AOI is kept.

The basemap (CARTO, OpenStreetMap, Esri imagery...) and the colormap of the single bands (named after
the water feature they enhance, e.g. *turbid* for the sediment plume of the Loire) can be changed with
the widgets; `basemap=` and `cmap=` set the initial ones.


In [ ]:
viewer = visual.ViewSpectral(preview.Rrs, bands=[490, 560, 665], reproject=True,
                             minmaxvalues=(0, 0.04), minmax=(0, 0.1),
                             basemap='CARTO Positron', cmap='cmo.turbid')
viewer.visu()

In [ ]:
if len(viewer.aoi_stream.data.get('xs', [])) > 0:
    aoi, aoi_source = viewer.get_geom(viewer.aoi_stream, crs=4326), 'drawn polygon'
else:
    aoi, aoi_source = default_aoi.to_crs(4326), AOI_FILE or 'default box'

print(f'AOI: {aoi_source}, bounds (lon/lat): {np.round(aoi.total_bounds, 4)}')
aoi

## 4. Crop

`grstbx.crop_l2a` cuts the full-resolution product to the bounding box of the AOI. With
`mask_outside=True`, the pixels outside the polygon(s) are then flagged as missing, following the GRS
conventions:

- float variables (`Rrs`, `BRDFg`, angles...) set to NaN;
- bit 0 (`nodata`) of `flags` raised;
- `mask` set to 1 (invalid pixel).

The ancillary data are cropped to the same bounding box enlarged by one coarse cell, so that they
can still be interpolated over the whole subset. The result is lazy: `.load()` reads the chunks
intersecting the AOI only.

In [ ]:
subset, subset_anc = grstbx.crop_l2a(l2a_prod, aoi, ancillary, mask_outside=MASK_OUTSIDE)
subset = subset.load()

print(f'subset: {subset.sizes["x"]} x {subset.sizes["y"]} pixels, '
      f'{subset.nbytes / 1e6:.0f} MB in memory, '
      f'{float(((subset.flags & 1) == 0).mean()) * 100:.0f}% of the pixels inside the AOI')
subset

True colour and GRS water mask of the subset:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 8), subplot_kw={'projection': proj})
plot_rgb(subset, axs[0], aoi=aoi)
axs[0].set_title('True colour')
(subset['mask'] == 0).where((subset.flags & 1) == 0).plot.imshow(
    ax=axs[1], cmap='Blues', vmin=0, vmax=1, add_colorbar=False)
axs[1].set_title('Valid water pixels (mask == 0, blue)');

## 5. Export

`grstbx.export_l2a` writes the subset with the exporters of GRS (`grs.output.L2aProduct`, the `grs`
package is needed), so it has the same layout and packing as the products of the processor:

- `Rrs`, `BRDFg`, `aot550`, angles, `dem`: `int16` with `scale_factor` / `add_offset`
  (e.g. 1e-5 sr$^{-1}$ for `Rrs`); `flags` and `mask` unchanged; other variables (e.g. `wind`) as float32;
- **NetCDF** (`.nc`): `<name>/<name>.nc` (main) and `<name>/<name>_anc.nc` (ancillary), zlib compression;
- **Zarr** (`.zarr`): `<name>.zarr` with the full resolution in group `0`, the overviews (pyramid) when
  the subset is larger than 512 pixels, and the group `ancillary`.

`grstbx.subset_path` builds the output path from the input name and `SITE`. The returned `out` is the
path to give to `grstbx.L2grs` (NetCDF product folder or Zarr store).

In [ ]:
out = grstbx.export_l2a(subset, grstbx.subset_path(FILE, ODIR, SITE, FMT), ancillary=subset_anc)

size = sum(os.path.getsize(opj(root, f)) for root, _, files in os.walk(out) for f in files)
print(f'{out}  ({size / 1e6:.2f} MB)')
print(sorted(os.listdir(out)))

### Optional: GeoTIFF of $R_{rs}$

A multi-band GeoTIFF (one band per wavelength, float32, NaN outside the AOI / on missing pixels) for
GIS software. Set `water_only=True` to keep only the valid water pixels (`mask == 0`).

In [ ]:
tif = grstbx.export_rrs_geotiff(subset, opj(ODIR, SITE, os.path.basename(out).removesuffix('.zarr') + '_Rrs.tif'))
print(tif, f'{os.path.getsize(tif) / 1e6:.2f} MB')

## 6. Check the exported product

Reopen the subset with `grstbx.L2grs`, as for any GRS product, and compare it with the data in memory.
The differences come from the `int16` packing only (at most half the `scale_factor`, i.e. 5e-6 sr$^{-1}$
for `Rrs`) and are zero when the input was itself a GRS product.

In [ ]:
dc = grstbx.L2grs([out])
dc.get_l2a_datacube(nodata_thresh=1.)
reloaded = dc.datacube.isel(time=0)

print('max |dRrs| :', float(abs(reloaded.Rrs - subset.Rrs).max()))
print('flags equal:', bool((reloaded.flags.values == subset.flags.values).all()), reloaded.flags.dtype)
print('flag statistics of the subset:')
print(dc.datacube[[v for v in dc.datacube.data_vars if v.startswith('flag_')]]
      .isel(time=0).to_pandas().round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': proj})
plot_rgb(reloaded, ax, aoi=aoi)
ax.set_title(f'Exported subset: {os.path.basename(out)}', fontsize=10);

## 7. Apply the AOI to a series of images

`grstbx.crop_and_export` opens, crops and exports a product in one call; existing outputs are skipped
(`overwrite=False`). Set `RUN_BATCH = True` to process every product matching `PATTERN` with the same AOI.

In [ ]:
RUN_BATCH = False
PATTERN = '/data/satellite/Sentinel-2/test/*T30TWT*/*.zarr'

if RUN_BATCH:
    for file in sorted(glob.glob(PATTERN)):
        print(grstbx.crop_and_export(file, aoi, grstbx.subset_path(file, ODIR, SITE, FMT),
                                     mask_outside=MASK_OUTSIDE))